In [20]:
import pandas as pd

df = pd.read_csv("data/train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [25]:
for col in df.columns:
    print(f"{col}: {df[col].isnull().sum()} missing values ({df[col].isnull().sum() / len(df) * 100:.2f}%)")

PassengerId: 0 missing values (0.00%)
Survived: 0 missing values (0.00%)
Pclass: 0 missing values (0.00%)
Name: 0 missing values (0.00%)
Sex: 0 missing values (0.00%)
Age: 177 missing values (19.87%)
SibSp: 0 missing values (0.00%)
Parch: 0 missing values (0.00%)
Ticket: 0 missing values (0.00%)
Fare: 0 missing values (0.00%)
Cabin: 687 missing values (77.10%)
Embarked: 2 missing values (0.22%)


## Data Dictionary

| Variable | Definition | Key |
|---|---|---|
| `survival` | Survival | 0 = No, 1 = Yes |
| `pclass` | Ticket class | 1 = 1st, 2 = 2nd, 3 = 3rd |
| `sex` | Sex | |
| `Age` | Age in years | |
| `sibsp` | Number of siblings or spouses aboard the Titanic | |
| `parch` | Number of parents or children aboard the Titanic | |
| `ticket` | Ticket number | |
| `fare` | Passenger fare | |
| `cabin` | Cabin number | |
| `embarked` | Port of embarkation | C = Cherbourg, Q = Queenstown, S = Southampton |

## Variable Notes

- **`pclass`:** A proxy for socio-economic status (SES): 1st = Upper, 2nd = Middle, 3rd = Lower.
- **`age`:** Age is fractional for children under 1. Estimated ages are written in the form `xx.5`.
- **`sibsp`:** The dataset defines family relations as follows:
  - Sibling = brother, sister, stepbrother, stepsister.
  - Spouse = husband or wife; mistresses and fiancés were ignored.
- **`parch`:** The dataset defines family relations as follows:
  - Parent = mother or father.
  - Child = daughter, son, stepdaughter, stepson.
  - Some children travelled only with a nanny, so their `parch` value is 0.
  
Taking a moment to first reflect on the features present in the data we can see there are 12 overall features pertaining to the personal details of the passenger (name, survival status, sex, age) as well as additional data about familial relations with whom they embarked (SibSp, Parch). It's worth noting that the Kaggle dataset description states that SibSp and Parch only count relations that were actually aboard the titanic, those that did not embark are exlcuded from this column. This needs to be asserted in exploration.

As well as the personal details relating to the passenger, there are also features relating to the ticket purchased such as PassengerId, Pclass, Ticket, Fare, Cabin and Embarked.

From the dataset information display we can see that the following columns contained NaN values: Age (18.87%), Cabin (77.1%) and Embarked (0.22%). Further exploration is required to determine a handling approach

## Anomalies and Cleaning

In [3]:
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
754,755,1,2,"Herman, Mrs. Samuel (Jane Laver)",female,48.0,1,2,220845,65.0000,NaN,S
433,434,0,3,"Kallio, Mr. Nikolai Erland",male,17.0,0,0,STON/O 2. 3101274,7.1250,NaN,S
674,675,0,2,"Watson, Mr. Ennis Hastings",male,NaN,0,0,239856,0.0000,NaN,S
499,500,0,3,"Svensson, Mr. Olof",male,24.0,0,0,350035,7.7958,NaN,S
54,55,0,1,"Ostby, Mr. Engelhart Cornelius",male,65.0,0,1,113509,61.9792,B30,C


In the samples above, we can observe the inconsistent format of the Ticket attribute, with 4/5 samples displaying an integer value while 1 example appears to be a string containing "STON/O 2." before the integer ticket number. Analysis is required to determine the meaning of this string component, my immediate hypothesis is that it somehow indicates the passenger boarded with the ticket of a relative but this is speculation and must be confirmed via analysis.

In [5]:
df.describe(include="object")

/tmp/ipykernel_92206/702825166.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


In the description of the object type columns I can see that all passenger names are unique, occuring no more than once.

Only two sexes are defined, with male being the most common (~65%).

The ticket feature appears to be non-unique for some entries, out of 891 entries, only 681 are unique (~76%). Further exploration is required to determine if these are anomalous or if some passengers are covered under a single ticket under specific circumstances like familial relations or organisations.

Cabin appears to have no-unique requirement, meaning most passengers shared a cabin with atleast one other passenger. To check for anomalies it's neccessary to analyse the number of passengers in each cabin.

Embarked has been counted with 3 unique values, each indicating the location of departure for the passenger. The kaggle dataset description lists these locations: C = Cherbourg, Q = Queenstown and S = Southampton.

In [33]:
df["PassengerId"] = df["PassengerId"].astype(str)
df["Survived"] = df["Survived"].astype(str)
df["Pclass"] = df["Pclass"].astype(str)
df.describe(include="object")

/tmp/ipykernel_92206/1751595833.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,PassengerId,Survived,Pclass,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,891,891,891,204,889
unique,891,2,3,891,2,681,147,3
top,1,0,3,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,549,491,1,577,7,4,644


In the previous cell I coerced the `PassengerId`, `Survived` and `Pclass` features to string type to ensure they are treated as categorical rather than numeric.

In [34]:
df.describe(include="number")

,Age,SibSp,Parch,Fare
count,714.000000,891.000000,891.000000,891.000000
mean,29.699118,0.523008,0.381594,32.204208
std,14.526497,1.102743,0.806057,49.693429
min,0.420000,0.000000,0.000000,0.000000
25%,20.125000,0.000000,0.000000,7.910400
50%,28.000000,0.000000,0.000000,14.454200
75%,38.000000,1.000000,0.000000,31.000000
max,80.000000,8.000000,6.000000,512.329200


From the numeric feature descriptions we can see the mean age is approximately 30 years old, with the 75th percentile at 38, meaning 75% of passengers are under 40, with some outliers such as the maximum, all the way up at 80.

The vast majoirty of passengers boarded without siblings, spouses, parents or children. For `SibSp`, the 75th percentile is `1`, for `Parch` it is `0`.

The fare feature shows huge variance between paid prices for tickets with the minimum being 0 and the maximum equal to ~512.33. Interestingly, despite the minimum being 0, only 25% of fares are under `7.9`, suggesting either a data entry error or that some passengers did indeed travel for free. This is pontentially anomalous and requires investigation alongside the `Pclass`, `Cabin` and `Ticket` fields for potential clues.


* SibSp & Parch: the Kaggle dataset description states that SibSp and Parch only count relations that were actually aboard the titanic, those that did not embark are exlcuded from this column. This needs to be asserted in exploration.
* Cabin: To check for anomalies it's neccessary to analyse the number of passengers in each cabin.
* Ticket: out of 891 entries, only 681 are unique (~76%). Further exploration is required to determine if these are anomalous or if some passengers are covered under a single ticket under specific circumstances like familial relations or organisations.
* Embarked: 3 counted unique values (S, C and ?) need to analyse and determine if the third is valid
* NaN analysis: Age, Cabin, embarked
* Ticket; string component meaning
* low/0 ticket fares: check with the `Pclass`, `Cabin` and `Ticket` fields for potential clues.